# 01_04 - Ubuntu Data Collection

## Mục tiêu

Notebook này thực hiện bước **Data Collection** cho source:

**Ubuntu**

Nguồn:

**OPUS Ubuntu v14.10**

Các nhiệm vụ:

1. Kiểm tra môi trường
2. Xác định project root
3. Cấu hình source
4. Download dataset
5. Convert dữ liệu sang DataFrame
6. Inspect raw data
7. Thống kê raw data
8. Xác định technology candidate
9. Tổng hợp audit summary
10. Lưu raw JSONL
11. Lưu raw Parquet
12. Lưu audit summary
13. Lưu metadata
14. Final verification
15. Ghi trạng thái notebook

> Lưu ý:
> - Notebook này không cleaning dữ liệu
> - Không deduplication
> - Không train/validation/test split
> - Không overwrite raw data
> - Technology candidate không đồng nghĩa với usable IT corpus
> - `usable_count` chưa được xác định


In [ ]:
import sys
import os
from pathlib import Path

print("Python:", sys.version)
print("Working directory:", os.getcwd())

In [ ]:
import datasets
import pandas as pd
import httpx

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("httpx:", httpx.__version__)

In [ ]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()


def find_project_root(start_path: Path) -> Path:
    """
    Tìm project root bằng cách kiểm tra:
    - data/
    - notebooks/ hoặc notebook/
    """
    candidates = [start_path] + list(start_path.parents)

    for path in candidates:
        if (
            (path / "data").is_dir()
            and (
                (path / "notebooks").is_dir()
                or (path / "notebook").is_dir()
            )
        ):
            return path

    raise FileNotFoundError(
        "Không tìm thấy project root. "
        "Hãy kiểm tra lại vị trí notebook."
    )


PROJECT_ROOT = find_project_root(CURRENT_DIR)

print("Project root:")
print(PROJECT_ROOT)

In [ ]:
SOURCE_NAME = "Ubuntu"

SOURCE_SHORT_NAME = "ubuntu"

SOURCE_URL = (
    "https://opus.nlpl.eu/Ubuntu"
)

DATASET_ID = (
    "OPUS-ubuntu-v14.10-eng-vie"
)

LANGUAGE_PAIR = "en-vi"

DOMAIN = "Software Localization"

DATASET_VERSION = "v14.10"

DOWNLOAD_METHOD = (
    "mtdata 0.4.3"
)

COLLECTION_DATE = pd.Timestamp.now().strftime("%Y-%m-%d")

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / SOURCE_SHORT_NAME
)

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Source:", SOURCE_NAME)
print("Source URL:", SOURCE_URL)
print("Dataset ID:", DATASET_ID)
print("Language pair:", LANGUAGE_PAIR)
print("Domain:", DOMAIN)
print("Dataset version:", DATASET_VERSION)
print("Download method:", DOWNLOAD_METHOD)
print("Raw directory:", RAW_DIR)

In [ ]:
import subprocess
import shutil

DATASET_ID = "OPUS-ubuntu-v14.10-eng-vie"

TRAIN_PARTS_DIR = RAW_DIR / "train-parts"

SOURCE_FILE = TRAIN_PARTS_DIR / f"{DATASET_ID}.eng"
TARGET_FILE = TRAIN_PARTS_DIR / f"{DATASET_ID}.vie"
REFERENCES_FILE = RAW_DIR / "references.bib"

dataset_already_downloaded = (
    SOURCE_FILE.is_file()
    and TARGET_FILE.is_file()
    and REFERENCES_FILE.is_file()
)

if dataset_already_downloaded:
    print("Dataset đã tồn tại.")
    print("Bỏ qua bước download lại.")

else:
    mtdata_executable = shutil.which("mtdata")

    if mtdata_executable is None:
        raise FileNotFoundError(
            "Không tìm thấy mtdata. "
            "Cần cài mtdata để tải Ubuntu lần đầu."
        )

    print("Dataset chưa hoàn chỉnh.")
    print("Bắt đầu download bằng mtdata...")

    command = [
        mtdata_executable,
        "get",
        "-l",
        "eng-vie",
        "-tr",
        DATASET_ID,
        "-o",
        str(RAW_DIR),
    ]

    result = subprocess.run(
        command,
        check=False,
        capture_output=True,
        text=True,
    )

    print("Return code:", result.returncode)
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)

    if result.returncode != 0:
        raise RuntimeError("mtdata download failed.")

required_files = [SOURCE_FILE, TARGET_FILE, REFERENCES_FILE]

print("\nFinal verification:")
for file_path in required_files:
    print(
        f"{file_path.name:50}"
        f"{'OK' if file_path.is_file() else 'MISSING'}"
    )

if not all(file_path.is_file() for file_path in required_files):
    raise FileNotFoundError("Ubuntu raw/provenance files chưa đầy đủ.")

print("\nDataset ID:", DATASET_ID)
print("Output directory:", RAW_DIR)

In [ ]:
TRAIN_PARTS_DIR = (
    RAW_DIR
    / "train-parts"
)

SOURCE_FILE = (
    TRAIN_PARTS_DIR
    / "OPUS-ubuntu-v14.10-eng-vie.eng"
)

TARGET_FILE = (
    TRAIN_PARTS_DIR
    / "OPUS-ubuntu-v14.10-eng-vie.vie"
)

if not SOURCE_FILE.is_file():
    raise FileNotFoundError(
        "Không tìm thấy English source file:\n"
        f"{SOURCE_FILE}"
    )

if not TARGET_FILE.is_file():
    raise FileNotFoundError(
        "Không tìm thấy Vietnamese target file:\n"
        f"{TARGET_FILE}"
    )

with open(
    SOURCE_FILE,
    "r",
    encoding="utf-8"
) as f:
    en_lines = [
        line.rstrip("\n")
        for line in f
    ]

with open(
    TARGET_FILE,
    "r",
    encoding="utf-8"
) as f:
    vi_lines = [
        line.rstrip("\n")
        for line in f
    ]

if len(en_lines) != len(vi_lines):
    raise ValueError(
        "English và Vietnamese không có cùng số dòng."
    )

df = pd.DataFrame(
    {
        "en": en_lines,
        "vi": vi_lines
    }
)

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
print("First 5 rows:")
display(df.head())

print("\nRandom 5 rows:")
display(
    df.sample(
        5,
        random_state=42
    )
)

In [ ]:
raw_count = len(df)

missing_values = df.isna().sum()

duplicate_count = df.duplicated().sum()

print("Raw count:", raw_count)

print("\nMissing values:")
display(missing_values)

print("\nDuplicate rows:")
print(duplicate_count)

print("\nColumn statistics:")

for column in df.columns:
    print(f"\n{column}")
    print(
        "Non-null:",
        df[column].notna().sum()
    )
    print(
        "Unique:",
        df[column].nunique(
            dropna=False
        )
    )

In [ ]:
df_tech_candidate = df.copy()

technology_candidate_count = len(
    df_tech_candidate
)

non_technology_count = (
    raw_count
    - technology_candidate_count
)

print("Raw rows:", raw_count)

print(
    "Technology candidate rows:",
    technology_candidate_count
)

print(
    "Non-technology rows:",
    non_technology_count
)

In [ ]:
audit_summary = {
    "source": SOURCE_NAME,
    "dataset_id": DATASET_ID,
    "dataset_version": DATASET_VERSION,
    "raw_count": int(raw_count),
    "candidate_count": int(
        technology_candidate_count
    ),
    "non_technology_count": int(
        non_technology_count
    ),
    "missing_values": {
        str(key): int(value)
        for key, value in missing_values.items()
    },
    "duplicate_count": int(
        duplicate_count
    ),
    "columns": [
        str(column)
        for column in df.columns
    ],
    "language_pair": LANGUAGE_PAIR,
    "mtdata_segments": 5056,
    "mtdata_errors": 0,
    "candidate_rule": (
        "All rows from the Ubuntu "
        "English-Vietnamese software localization "
        "corpus are retained as technology candidates. "
        "This does not establish final usable IT status."
    )
}

audit_summary

In [ ]:
raw_jsonl_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.jsonl"
)
raw_parquet_path = RAW_DIR / f"{SOURCE_SHORT_NAME}_raw.parquet"
audit_path = RAW_DIR / "audit_summary.json"
metadata_path = RAW_DIR / "metadata.json"

# Phase 01 tạo snapshot RAW một lần; không ghi đè artifact đã có.
phase_01_output_paths = [raw_jsonl_path, raw_parquet_path, audit_path, metadata_path]
existing_outputs = [path for path in phase_01_output_paths if path.exists()]
missing_outputs = [path for path in phase_01_output_paths if not path.exists()]
if existing_outputs and missing_outputs:
    raise RuntimeError(
        "Phát hiện RAW snapshot chưa đầy đủ; không được ghi đè hay tiếp tục. \n"
        f"Existing: {[str(path) for path in existing_outputs]}\n"
        f"Missing: {[str(path) for path in missing_outputs]}"
    )

write_raw_snapshot = not existing_outputs
if write_raw_snapshot:
    print("No existing RAW snapshot found; creating a new immutable snapshot.")
else:
    print("Complete RAW snapshot already exists; preserving it and skipping writes.")

if write_raw_snapshot:
    df.to_json(
        raw_jsonl_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_jsonl_path)

In [ ]:
raw_parquet_path = (
    RAW_DIR
    / f"{SOURCE_SHORT_NAME}_raw.parquet"
)

if write_raw_snapshot:
    df.to_parquet(
        raw_parquet_path,
        index=False
    )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(raw_parquet_path)

In [ ]:
import json

audit_path = (
    RAW_DIR
    / "audit_summary.json"
)

if write_raw_snapshot:
    with open(
        audit_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            audit_summary,
            f,
            ensure_ascii=False,
            indent=2
        )

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(audit_path)

In [ ]:
metadata = {
    "source": SOURCE_NAME,
    "source_url": SOURCE_URL,
    "version_revision": DATASET_VERSION,
    "collection_date": COLLECTION_DATE,
    "download_method": DOWNLOAD_METHOD,
    "language_pair": LANGUAGE_PAIR,
    "domain": DOMAIN,
    "subcategory": 'Ubuntu software localization',
    "raw_count": int(raw_count),
    "candidate_count": int(technology_candidate_count),
    "usable_count": None,
    "notes": 'Raw English-Vietnamese Ubuntu localization corpus collected with MTData. Final IT usability is determined in later pipeline phases.',
}

metadata_path = RAW_DIR / "metadata.json"
if write_raw_snapshot:
    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Saved:" if write_raw_snapshot else "Preserved existing:")
print(metadata_path)


In [ ]:
REFERENCES_FILE = (
    RAW_DIR
    / "references.bib"
)

expected_files = [
    SOURCE_FILE,
    TARGET_FILE,
    REFERENCES_FILE,
    raw_jsonl_path,
    raw_parquet_path,
    audit_path,
    metadata_path
]

verification_results = {}

for file_path in expected_files:
    verification_results[
        file_path.name
    ] = file_path.is_file()

print("Final verification:\n")

for file_name, exists in verification_results.items():
    print(
        f"{file_name:55}"
        f"{'OK' if exists else 'MISSING'}"
    )

verification_passed = all(
    verification_results.values()
)

print(
    "\nVerification passed:",
    verification_passed
)

# Data Collection Status

Source:

**Ubuntu — OPUS v14.10**

Dataset ID:

**OPUS-ubuntu-v14.10-eng-vie**

| Metric | Value |
|---|---:|
| MTData segments | 5,056 |
| MTData errors | 0 |
| Raw rows | 5,056 |
| Technology candidate rows | 5,056 |
| Non-technology rows | 0 |
| Usable IT rows | TBD |

## Completed in this notebook

- [x] Environment checked
- [x] Project root identified
- [x] Source configured
- [x] Ubuntu English-Vietnamese corpus downloaded
- [x] MTData reported 5,056 segments
- [x] MTData reported 0 errors
- [x] Raw parallel files converted to DataFrame
- [x] Raw schema inspected
- [x] Raw statistics recorded
- [x] Technology candidate identified
- [x] Raw JSONL saved
- [x] Raw Parquet saved
- [x] Audit summary saved
- [x] Metadata saved
- [x] Raw source files verified
- [x] Provenance file verified
- [x] Output files verified

## Not completed in this notebook

- [ ] Full language check
- [ ] Alignment check
- [ ] Data cleaning
- [ ] Deduplication
- [ ] Quality/noise assessment
- [ ] IT subdomain classification
- [ ] Final usable IT count
- [ ] Train / validation / test split

## Interpretation

Ubuntu is a software-localization corpus and is therefore retained
as a technology candidate at the source-collection stage.

The downloaded source contains:

`5,056` English-Vietnamese segments

with:

`0` errors reported by MTData.

However:

`technology_candidate_count != usable_count`

The final usable IT corpus must only be determined after the
subsequent audit, cleaning and IT-filtering stages.


> Raw data under `data/raw/` must remain unchanged.
>
> This notebook does not produce the final IT corpus.
